In [24]:
from sat_com_adapter.adapters import NetworkXAdapter
from sat_com_adapter.networkx_builder.networkx_builder import NetworkxBuilder

plain_config_walker = {
  "simulation_name": "Single Shell Simulation",
  "start_date": "2026-01-01 00:00:00.000000",
  "end_date": "2026-01-01 00:00:10.000000",
  "movement_model": "pyorbital",
  "distance_model": "sklearn",
  "ground_objects_properties": [
    {
      "identifier": "OneWeb Ground Station",
      "data_file": "./configurations/ground_stations.txt",
      "type": "ground_station",
      "connectivity_properties": {
        "elevation_above_horizon": 20,
        "ground_to_space_connections_strategy": "everything-visible"
      },
    },

  ],
  "walker_shells": [
    {
      "type": "delta",
      "constellation_property": {
        "identifier": "Andromeda Walker",
        "amount_of_orbit_plane": 12,
        "amount_of_satellite_per_orbit_plane": 22,
        "inclination": 70.0,
        "phase_difference_between_satellites": True,
        "mean_revolution_per_day": 15.0
      },
      "orbital_connectivity_property": {
        "adjacent_inter_satellite_shifting": 0,
        "maximum_inter_satellite_count": 4,
        "maximum_inter_satellite_range_distance": 1000,
        "maximum_ground_station_range": 5000,
        "maximum_user_terminal_range": 1000,
        "maximum_connected_ground_object": 10000,
        "maximum_connected_user_terminal": 1000,
        "maximum_connected_ground_station": 10
      },
      "ground_object_white_list": []
    }
  ]
}

    # {
    #   "identifier": "User Terminal",
    #   "data_file": "./configurations/user_terminal.txt",
    #   "type": "user_terminal",
    #   "connectivity_properties": {
    #     "elevation_above_horizon": 25,
    #     "ground_to_space_connections_strategy": "best-angle-until-disconnection"
    #   },
    # }


In [25]:
from sat_com_builder.configuration_manager import BaseConfigurationManager
from sat_com_builder.models import SimulationProperty

In [26]:
simulation_properties = SimulationProperty(**plain_config_walker)
base_config_manager = BaseConfigurationManager(simulation_property=simulation_properties)

simulation_manager = base_config_manager.load_simulation()


In [27]:
def record_rx_state():
    networkx_builder = NetworkxBuilder(
        simulation_manager=simulation_manager,
        export_positions=True, export_link_length=True
    )
    networkx_builder.add_topology_objects()
    networkx_builder.add_links()

    networkx_adapter = NetworkXAdapter(
        simulation_manager=simulation_manager
    )
    networkx_adapter.set_networkx_builder(networkx_builder)

    networkx_adapter.adapt(
        output_directory=f"network_states/topology_at_{simulation_manager.time_manager.current_time.strftime('%Y_%m_%d_%H_%M_%S')}.json",
    )

simulation_manager.time_manager.register_action(record_rx_state)


In [ ]:
%%sql


In [28]:
simulation_manager.time_manager.tick_until_the_end(
    ticking_time_in_seconds=1
)